# P7 检测器 A：风格特征

In [ ]:
import os, re, sys, random
if not os.path.exists("P7_A_predict.py") and os.path.exists("../P7_A_predict.py"):
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import sklearn
import xgboost as xgb
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, f1_score, roc_curve
from sklearn.model_selection import GroupShuffleSplit, train_test_split

from P7_A_predict import extract_features, featurize_many, truncate_words, FEATURE_NAMES, WINDOW_WORDS

SEED = 42
random.seed(SEED); np.random.seed(SEED)
N_PER_SET = 1000
HUMAN_COLOR, AI_COLOR = "#3B6FD4", "#F28E2B"

## 1. 数据

In [ ]:
def balance(df, n, seed=SEED):
    k = min(n // 2, int(df.label.sum()), int((1 - df.label).sum()))
    out = pd.concat([df[df.label == 0].sample(k, random_state=seed),
                     df[df.label == 1].sample(k, random_state=seed)])
    return out.sample(frac=1, random_state=seed).reset_index(drop=True)

def clean(df):
    df = df.copy()
    df["text"] = df["text"].astype(str)
    df = df[df.text.str.split().str.len() >= WINDOW_WORDS]
    df["text"] = df["text"].map(truncate_words)
    return df.reset_index(drop=True)

def load_hc3():
    from datasets import load_dataset
    try:
        ds = load_dataset("Hello-SimpleAI/HC3", "all", trust_remote_code=True)["train"]
    except (TypeError, ValueError):
        ds = load_dataset("Hello-SimpleAI/HC3", "all")["train"]
    rows = []
    for qid, ex in enumerate(ds):
        rows += [(qid, t, 0) for t in ex["human_answers"]]
        rows += [(qid, t, 1) for t in ex["chatgpt_answers"]]
    return clean(pd.DataFrame(rows, columns=["qid", "text", "label"]))

def load_mage(n=N_PER_SET * 4):
    from datasets import load_dataset
    df = load_dataset("yaful/MAGE", split="test").to_pandas()
    df["label"] = 1 - df["label"]    # 原标签 1 = 人类，改成 1 = AI
    return clean(df[["text", "label"]])

def load_raid(n=N_PER_SET * 8):
    from datasets import load_dataset
    ds = load_dataset("liamdugan/raid", split="train", streaming=True).shuffle(seed=SEED, buffer_size=10000)
    rows = []
    for ex in ds:
        if ex["attack"] != "none":
            continue
        rows.append((ex["generation"], 0 if ex["model"] == "human" else 1, ex.get("source_id", ex.get("id"))))
        if len(rows) >= n:
            break
    return clean(pd.DataFrame(rows, columns=["text", "label", "group"]))

def load_csv_set(path):
    return clean(pd.read_csv(path)[["text", "label"]])

In [ ]:
hc3 = load_hc3()
print("HC3:", len(hc3), "条，AI 占", round(hc3.label.mean(), 2))

# 按问题划分，同一个问题的回答不会同时出现在两个集合里
tr_idx, rest_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED).split(hc3, groups=hc3.qid))
train_all, rest = hc3.iloc[tr_idx], hc3.iloc[rest_idx]
va_idx, te_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED).split(rest, groups=rest.qid))
val_all, test_all = rest.iloc[va_idx], rest.iloc[te_idx]
assert not (set(train_all.qid) & set(val_all.qid)) and not (set(train_all.qid) & set(test_all.qid))

train, val, test = balance(train_all, 6000), balance(val_all, 2000), balance(test_all, 2000)
print("训练/验证/测试:", len(train), len(val), len(test))

test_sets = {"HC3": test}
for name, fn in [("MAGE", load_mage), ("RAID (subset)", load_raid)]:
    try:
        test_sets[name] = balance(fn(), N_PER_SET)
        print(name, len(test_sets[name]), "条")
    except Exception as e:
        print(name, "加载失败:", repr(e)[:120])

## 2. TF-IDF + 逻辑回归

In [ ]:
tfidf_log = []
best_tfidf, best_auc = None, -1
for ngram in [(1, 1), (1, 2)]:
    for C in [0.3, 1, 3, 10]:
        pipe = Pipeline([
            ("tfidf", TfidfVectorizer(ngram_range=ngram, min_df=3, max_features=50000, sublinear_tf=True)),
            ("lr", LogisticRegression(C=C, max_iter=1000)),
        ])
        pipe.fit(train.text, train.label)
        auc = roc_auc_score(val.label, pipe.predict_proba(val.text)[:, 1])
        tfidf_log.append({"ngram_range": ngram, "C": C, "val_auroc": round(auc, 4)})
        if auc > best_auc:
            best_tfidf, best_auc = pipe, auc
print(pd.DataFrame(tfidf_log).to_string(index=False))

## 3. 手工特征 + XGBoost

In [ ]:
def feats(df):
    return featurize_many(df.text)

X_train, X_val = feats(train), feats(val)
y_train, y_val = train.label.values, val.label.values
X_test = {name: feats(df) for name, df in test_sets.items()}
print(X_train.shape)

In [ ]:
def make_xgb(depth, lr, n):
    return xgb.XGBClassifier(max_depth=depth, learning_rate=lr, n_estimators=n,
                             subsample=0.8, colsample_bytree=0.8, eval_metric="logloss",
                             random_state=SEED, n_jobs=-1)

xgb_log = []
best_xgb, best_xgb_auc = None, -1
for depth in [3, 5, 7]:
    for lr in [0.05, 0.1]:
        for n in [200, 400]:
            m = make_xgb(depth, lr, n).fit(X_train, y_train)
            auc = roc_auc_score(y_val, m.predict_proba(X_val)[:, 1])
            xgb_log.append({"max_depth": depth, "learning_rate": lr, "n_estimators": n, "val_auroc": round(auc, 4)})
            if auc > best_xgb_auc:
                best_xgb, best_xgb_auc, best_params = m, auc, (depth, lr, n)
print(pd.DataFrame(xgb_log).sort_values("val_auroc", ascending=False).head(6).to_string(index=False))
print("最佳参数:", best_params)

## 4. 评测

In [ ]:
def tpr_at_fpr(y, s, target=0.01):
    fpr, tpr, _ = roc_curve(y, s)
    return float(tpr[fpr <= target].max())

def best_f1_threshold(y, s):
    grid = np.linspace(0.05, 0.95, 91)
    return float(grid[int(np.argmax([f1_score(y, s >= t) for t in grid]))])

def metrics(y, s, thr):
    return {"auroc": roc_auc_score(y, s), "f1": f1_score(y, s >= thr), "tpr_at_1fpr": tpr_at_fpr(y, s)}

# 阈值只在 HC3 验证集上选一次
thr_xgb = best_f1_threshold(y_val, best_xgb.predict_proba(X_val)[:, 1])
thr_tfidf = best_f1_threshold(y_val, best_tfidf.predict_proba(val.text)[:, 1])
print("阈值:", thr_xgb, thr_tfidf)

rows, pred_rows = [], []
for name, df in test_sets.items():
    s_x = best_xgb.predict_proba(X_test[name])[:, 1]
    s_t = best_tfidf.predict_proba(df.text)[:, 1]
    for det, s, thr in [("A", s_x, thr_xgb), ("A_tfidf_lr", s_t, thr_tfidf)]:
        rows.append({"dataset": name, "detector": det, **metrics(df.label.values, s, thr)})
    pred_rows += [{"dataset": name, "id": i, "gold": g, "pred": int(p >= thr_xgb), "score": p}
                  for i, (g, p) in enumerate(zip(df.label.values, s_x))]

results = pd.DataFrame(rows)
print(results.round(3).to_string(index=False))

## 5. 跨数据集泛化与特征重要性

In [ ]:
splits = {"HC3": (train, test)}
for name, df in test_sets.items():
    if name == "HC3":
        continue
    if "group" in df.columns and df["group"].notna().all() and df["group"].nunique() > 5:
        idx_a, idx_b = next(GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=SEED).split(df, groups=df["group"]))
        a, b = df.iloc[idx_a], df.iloc[idx_b]
    else:
        a, b = train_test_split(df, test_size=0.5, random_state=SEED, stratify=df.label)
    splits[name] = (a.reset_index(drop=True), b.reset_index(drop=True))

models = {name: make_xgb(*best_params).fit(feats(a), a.label.values) for name, (a, b) in splits.items()}
pool = pd.concat([a for a, b in splits.values()])
models["Pooled"] = make_xgb(*best_params).fit(feats(pool), pool.label.values)
test_feats = {name: feats(b) for name, (a, b) in splits.items()}
matrix = pd.DataFrame(
    {te: {tr: roc_auc_score(splits[te][1].label.values, models[tr].predict_proba(test_feats[te])[:, 1])
          for tr in models} for te in splits})
matrix.index.name = "train \\ test"
print(matrix.round(2))
matrix.round(4).to_csv("P7_A_generalization_matrix.csv")

In [ ]:
imp = pd.Series(best_xgb.feature_importances_, index=FEATURE_NAMES).sort_values(ascending=False)
top = imp.head(12)[::-1]
fig, ax = plt.subplots(figsize=(6, 4.5))
ax.barh(top.index, top.values, color="#666666")
ax.set_xlabel("XGBoost feature importance")
plt.tight_layout(); plt.savefig("P7_A_feature_importance.png", dpi=200); plt.show()

Xz = (X_train - X_train.mean(0)) / (X_train.std(0) + 1e-9)
names = list(imp.head(8).index)
idx = [FEATURE_NAMES.index(n) for n in names]
mh, ma = Xz[y_train == 0][:, idx].mean(0), Xz[y_train == 1][:, idx].mean(0)
fig, ax = plt.subplots(figsize=(6.5, 4))
pos = np.arange(len(names))
ax.bar(pos - 0.2, mh, 0.4, color=HUMAN_COLOR, label="Human")
ax.bar(pos + 0.2, ma, 0.4, color=AI_COLOR, label="AI")
ax.set_xticks(pos); ax.set_xticklabels(names, rotation=40, ha="right"); ax.axhline(0, color="k", lw=0.5)
ax.set_ylabel("mean (standardised)"); ax.legend(); plt.tight_layout()
plt.savefig("P7_A_feature_direction.png", dpi=200); plt.show()

## 6. 保存模型

In [ ]:
os.makedirs("models", exist_ok=True); os.makedirs("requirements", exist_ok=True)
joblib.dump({"xgb": best_xgb, "threshold": thr_xgb, "tfidf_lr": best_tfidf}, "models/P7_A_model.pkl", compress=3)
print("模型大小 MB:", round(os.path.getsize("models/P7_A_model.pkl") / 1e6, 2))

with open("requirements/P7_A.txt", "w") as f:
    f.write(f"numpy=={np.__version__}\nscikit-learn=={sklearn.__version__}\nxgboost=={xgb.__version__}\njoblib=={joblib.__version__}\n")

results.round(4).to_csv("P7_A_metrics.csv", index=False)
pd.DataFrame(pred_rows).to_csv("P7_A_preds.csv", index=False)

In [ ]:
import importlib, P7_A_predict
importlib.reload(P7_A_predict)

AI_SAMPLE = ("Artificial intelligence has the potential to transform education by providing personalized learning experiences. "
             "However, it is important to consider the ethical implications, including privacy concerns and the risk of over-reliance on technology. "
             "Overall, a balanced approach that combines human guidance with intelligent tools can help students achieve better outcomes.")
HUMAN_SAMPLE = ("honestly i didnt expect the exam to be that long lol. spent way too much time on q3 and had to rush the last part, "
                "so we will see how it goes. my roommate said she finished with 20 mins left but she always says that. anyway im going "
                "for bubble tea later if anyone is free, i need to get out of the library for a bit")
for t in [AI_SAMPLE, HUMAN_SAMPLE, "Hi"]:
    print(P7_A_predict.predict(t))

## 7. 新生成器、攻击版

In [ ]:
EXTRA = {
    # "New generators": "路径.csv",
    # "After attack": "路径.csv",
}
extra_rows = []
for name, path in EXTRA.items():
    df = load_csv_set(path)
    s_x = best_xgb.predict_proba(feats(df))[:, 1]
    s_t = best_tfidf.predict_proba(df.text)[:, 1]
    for det, s, thr in [("A", s_x, thr_xgb), ("A_tfidf_lr", s_t, thr_tfidf)]:
        extra_rows.append({"dataset": name, "detector": det, **metrics(df.label.values, s, thr)})
    pred_rows += [{"dataset": name, "id": i, "gold": g, "pred": int(p >= thr_xgb), "score": p}
                  for i, (g, p) in enumerate(zip(df.label.values, s_x))]
if extra_rows:
    results = pd.concat([results, pd.DataFrame(extra_rows)], ignore_index=True)
    results.round(4).to_csv("P7_A_metrics.csv", index=False)
    pd.DataFrame(pred_rows).to_csv("P7_A_preds.csv", index=False)
    print(results.round(3).to_string(index=False))